# Quickstart: OMOP tables and a governed cohort count from public ISIC data

This notebook shows the two things an investigator uses here:

1. the **OMOP tables** this project builds from dermoscopy archive metadata;
2. the **governed cohort service**, which turns a versioned phenotype into counts with small
   cells suppressed.

It runs on **public metadata only**: ISIC Archive collection 406 (PAD-UFES-20, CC-BY,
attribution: Federal University of Espírito Santo). It downloads no images, needs no login,
and takes about a minute. It is a portfolio project, not a clinical data source.

The workspace copy covers six sources and the full pipeline (see `docs/data_dictionary.md`).
This notebook rebuilds one source locally with the same code.

Install from the repository root: `pip install -e ".[cohort]"`.

In [ ]:
import os
import tempfile
from datetime import date
from pathlib import Path

import pandas as pd

from cdm.cohort import Approvals, AuditLog, CohortService, Refused, duckdb_engine
from cdm.omop import build_tables, check_tables, load_concept_map, public_concepts, silver_from_isic
from cdm.sources import collection_records, flatten

REPO = Path.cwd() if (Path.cwd() / "phenotypes").exists() else Path.cwd().parent
COLLECTION, SOURCE = 406, "pad_ufes"
# CI runs this notebook offline on a synthetic stand-in for the ISIC response.
OFFLINE = os.environ.get("CDM_QUICKSTART_OFFLINE") == '1'

## 1. Fetch the collection's metadata from the ISIC API

In [ ]:
def synthetic_records(n=60):
    diagnoses = ["Nevus", "Basal cell carcinoma", "Melanoma, NOS", "Seborrheic keratosis"]
    return [
        {"isic_id": f"ISIC_SYN{i:04d}", "copyright_license": "CC-BY", "attribution": "synthetic",
         "files": {"full": {"url": ""}},
         "metadata": {"acquisition": {"image_type": "clinical: close-up"},
                      "clinical": {"patient_id": f"IP_{i // 2}", "lesion_id": f"IL_{i}",
                                   "sex": ["male", "female"][i % 2], "age_approx": 30 + i % 50,
                                   "anatom_site_1": "Head and neck", "anatom_site_2": "Head",
                                   "diagnosis_3": diagnoses[i % 4],
                                   "diagnosis_confirm_type": "histopathology" if i % 3 else
                                   "single image expert consensus"}}}
        for i in range(n)
    ]


records = synthetic_records() if OFFLINE else collection_records(COLLECTION)
silver = silver_from_isic(flatten(r, SOURCE, COLLECTION) for r in records)
print(len(silver), "images;", silver["patient_id"].nunique(), "patients")
silver[["isic_id", "patient_id", "lesion_id", "diagnosis_3", "anatom_site_2", "license"]].head()

## 2. Build the OMOP tables and run the quality gates

`config/omop_concepts.csv` maps each source value to an OMOP concept ID. Here the concept
table carries IDs and Athena links only: SNOMED names are not redistributed. The same seven
gates that guard the workspace pipeline run here.

In [ ]:
cmap = load_concept_map(REPO / "config" / "omop_concepts.csv")
tables = build_tables(silver, cmap, public_concepts(cmap), "quickstart", "public", date.today())
tables = {name: frame.infer_objects() for name, frame in tables.items()}
gates = check_tables(silver, tables, cmap)
print({name: len(frame) for name, frame in tables.items()})
pd.DataFrame([g.__dict__ for g in gates])

In [ ]:
tables["condition_occurrence"][["person_id", "condition_concept_id", "condition_source_value",
                                 "condition_status_source_value"]].head()

## 3. Ask a cohort question through the governed service

A phenotype is a versioned YAML file in `phenotypes/`. The service refuses to run one that a
reviewer has not approved, in its exact current form. Here you play the reviewer, in a
scratch approvals file that stays on your machine.

In [ ]:
scratch = Path(tempfile.mkdtemp())
engine, names = duckdb_engine(tables)
service = CohortService(engine, names, Approvals(scratch / "approvals.jsonl"),
                        AuditLog(scratch / "audit.jsonl"), source="quickstart")
phenotype = REPO / "phenotypes" / "melanoma_any" / "v1.yaml"
try:
    service.run(phenotype, requester="quickstart user")
except Refused as refusal:
    print("Refused:", refusal)

In [ ]:
service.decide(phenotype, "approve", reviewer="you (scratch copy)", reason="quickstart")
result = service.run(phenotype, requester="quickstart user")
print(result.note)
pd.DataFrame(result.attrition)

In [ ]:
pd.DataFrame(result.breakdown)

Every refusal, decision and run went to the audit log. Each entry carries the hash of the
one before, so an edit breaks the chain:

In [ ]:
pd.DataFrame(service.audit.entries())[["seq", "event", "phenotype", "requester", "hash"]]

## Next

- `docs/data_dictionary.md`: every table and column, with null rates from the workspace build.
- `phenotypes/README.md`: the phenotype language and the request, approve and run workflow.
- `docs/governance.md`: access tiers, small-cell rules, escalation.
- To ask a question of the full data, open a request with the issue form.